Performance in Spark comes down to three things: how data is partitioned, how often it shuffles, and how well Catalyst can optimize your query plan. This notebook covers all three — from controlling partition count and layout, to reading execution plans, to understanding the optimizations Catalyst applies automatically.

## Setup

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, count, date_format, month, year

spark = (
    SparkSession.builder
    .appName("PartitioningShufflesCatalyst")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")   # reduced from 200 for local demo
    .getOrCreate()
)

DATA = os.path.join(os.path.dirname(os.path.abspath("12-partitioning-shuffles-catalyst.ipynb")), "data")
OUT  = os.path.join(DATA, "out")

card_txns = spark.read.schema(StructType([
    StructField("txn_id",      StringType(),       False),
    StructField("card_id",     StringType(),       False),
    StructField("customer_id", StringType(),       False),
    StructField("amount",      DecimalType(18,2),  False),
    StructField("merchant",    StringType(),       True),
    StructField("category",    StringType(),       True),
    StructField("txn_ts",      TimestampType(),    False),
    StructField("status",      StringType(),       False),
    StructField("is_fraud",    BooleanType(),      False),
])).parquet(f"{DATA}/card_transactions")

customers = spark.read.option("header","true").schema(StructType([
    StructField("customer_id",   StringType(),    False),
    StructField("full_name",     StringType(),    False),
    StructField("email",         StringType(),    True),
    StructField("phone",         StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("state",         StringType(),    True),
    StructField("date_of_birth", DateType(),      True),
    StructField("kyc_verified",  BooleanType(),   False),
    StructField("created_at",    TimestampType(), False),
])).csv(f"{DATA}/customers")

loan_accounts = spark.read.option("multiLine","true").schema(StructType([
    StructField("loan_id",       StringType(),       False),
    StructField("customer_id",   StringType(),       False),
    StructField("loan_type",     StringType(),       False),
    StructField("principal",     DecimalType(18,2),  False),
    StructField("interest_rate", DoubleType(),       False),
    StructField("tenure_months", IntegerType(),      False),
    StructField("disbursed_on",  DateType(),         False),
    StructField("status",        StringType(),       False),
])).json(f"{DATA}/loan_accounts")

payments = spark.read.format("delta").load(f"{DATA}/payments")

print(f"card_transactions : {card_txns.count()} rows, {card_txns.rdd.getNumPartitions()} partitions")
print(f"customers         : {customers.count()} rows, {customers.rdd.getNumPartitions()} partitions")

## Partitions — The Unit of Parallelism

Each partition is processed by exactly one task on one executor core. Too few partitions under-utilises the cluster; too many creates scheduling overhead and tiny tasks.

**Rules of thumb:**
- Target 128 MB–256 MB per partition for large datasets
- For local mode: 2–4× the number of CPU cores
- After a shuffle: controlled by `spark.sql.shuffle.partitions` (default 200)

In [ ]:
# Inspect current partition counts
print("Default parallelism :", spark.sparkContext.defaultParallelism)
print("Shuffle partitions  :", spark.conf.get("spark.sql.shuffle.partitions"))
print()
print("card_txns partitions:", card_txns.rdd.getNumPartitions())
print("customers partitions:", customers.rdd.getNumPartitions())

# Check partition sizes (row count per partition)
from pyspark.sql.functions import spark_partition_id

card_txns.withColumn("pid", spark_partition_id()) \
         .groupBy("pid").count() \
         .orderBy("pid").show()

## `repartition` vs `coalesce`

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Shuffle | Full shuffle | No shuffle (narrow) |
| Direction | Increase or decrease | Decrease only |
| Output balance | Evenly distributed | Possibly unequal sizes |
| When to use | Before a heavy join/aggregation | Before writing small output |

`repartition(n, col)` — hash-partition by a column, ensuring all rows with the same key land in the same partition. Use before a join on that column to avoid a shuffle at join time.

In [ ]:
# repartition — increase to 16 and balance evenly (causes a shuffle)
repart = card_txns.repartition(16)
print(f"After repartition(16): {repart.rdd.getNumPartitions()} partitions")

# repartition by column — all transactions for the same customer land together
repart_by_cust = card_txns.repartition(8, col("customer_id"))
print(f"After repartition(8, customer_id): {repart_by_cust.rdd.getNumPartitions()} partitions")

# Verify: each partition should only contain a subset of customer_ids
repart_by_cust.withColumn("pid", spark_partition_id()) \
              .groupBy("pid") \
              .agg(count("txn_id").alias("rows"),
                   count_distinct("customer_id").alias("unique_customers")) \
              .orderBy("pid").show()

In [ ]:
from pyspark.sql.functions import countDistinct

# coalesce — reduce to 1 partition without a shuffle (for small CSV exports)
single = card_txns.filter(col("is_fraud") == True).coalesce(1)
print(f"Fraud txns coalesced: {single.rdd.getNumPartitions()} partition, {single.count()} rows")

## Partitioned Storage — `partitionBy`

Writing with `partitionBy` creates a directory hierarchy on disk — downstream readers that filter on the partition column skip irrelevant directories entirely (**partition pruning**).

```
data/card_transactions_partitioned/
    month_key=2024-01/   part-00000.parquet
    month_key=2024-02/   part-00000.parquet
    …
```

Choosing the right partition column matters: high-cardinality columns (like `customer_id`) create too many small files; prefer low-to-medium cardinality (status, month, category).

In [ ]:
# Add month_key column then write partitioned by it
card_txns_with_month = card_txns.withColumn(
    "month_key", date_format(col("txn_ts"), "yyyy-MM")
)

(
    card_txns_with_month
    .write.mode("overwrite")
    .partitionBy("month_key")
    .parquet(f"{OUT}/card_txns_by_month")
)

# Read back — Spark prunes to only the matching partition directory
txns_jan = spark.read.parquet(f"{OUT}/card_txns_by_month/month_key=2024-01")
print(f"January transactions: {txns_jan.count()} rows")

# With partition discovery — month_key becomes a column automatically
all_months = spark.read.parquet(f"{OUT}/card_txns_by_month")
all_months.groupBy("month_key").count().orderBy("month_key").show()

## What Triggers a Shuffle?

![Shuffle Anatomy](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-shuffle-anatomy.svg)

A shuffle happens whenever Spark must redistribute data across partitions. Each shuffle:
1. **Writes** intermediate data to local disk (shuffle write)
2. **Transfers** it over the network to target executors
3. **Reads** and deserialises it on the receiving side (shuffle read)

Operations that always trigger a shuffle:
`groupBy` · `join` · `distinct` · `orderBy` · `repartition` · `union` (distinct)

Operations that never shuffle:
`filter` · `select` · `withColumn` · `map` · `flatMap` · `coalesce`

In [ ]:
# Count the Exchange operators in a plan — each Exchange = one shuffle
from pyspark.sql.functions import count, col

agg_df = card_txns.groupBy("customer_id", "category").agg(count("txn_id").alias("n"))

# explain() — look for "Exchange" nodes which mark shuffle boundaries
agg_df.explain()

In [ ]:
# Join plan — shows two Exchange operators (one per side of the join)
joined = card_txns.join(customers, on="customer_id", how="inner")
print("=== JOIN PLAN ===")
joined.explain()

## Catalyst Optimizer

![Catalyst Phases](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-catalyst-phases.svg)

Every query — SQL or DataFrame API — passes through four Catalyst phases before execution:

1. **Analysis** — resolve column names and types against the schema catalog
2. **Logical Optimization** — apply rule-based rewrites: predicate pushdown, constant folding, column pruning
3. **Physical Planning** — choose join strategies, decide sort orders, estimate costs
4. **Code Generation** — Tungsten generates optimised JVM bytecode for tight inner loops

## Predicate Pushdown

Catalyst moves filter conditions as close to the data source as possible — ideally into the file reader itself. For Parquet and ORC this means row groups are skipped without being decoded.

In [ ]:
# Predicate pushdown — the filter is pushed into the Parquet reader
pushed = card_txns.filter(col("status") == "SUCCESS").select("txn_id", "amount")

print("=== WITH PREDICATE PUSHDOWN ===")
pushed.explain(mode="extended")
# Look for 'PushedFilters' in the FileScan node — the filter ran at read time

## Column Pruning

Catalyst tracks which columns are actually needed and tells the file reader to skip the rest. For columnar formats (Parquet, ORC) this is especially valuable — unread columns are never decoded.

In [ ]:
# Column pruning — only txn_id and amount are needed
pruned = card_txns.select("txn_id", "amount")

print("=== COLUMN PRUNING ===")
pruned.explain(mode="extended")
# Look for 'ReadSchema' in the FileScan node — only 2 of 9 columns are read

## `explain()` Modes

| Mode | What you see |
|---|---|
| `explain()` or `"simple"` | Physical plan only — quickest overview |
| `"extended"` | Unresolved → analyzed → optimized → physical plans |
| `"codegen"` | Generated JVM code (verbose, rarely needed) |
| `"cost"` | Physical plan with estimated row counts and sizes |
| `"formatted"` | Physical plan in readable tree format with stats |

In [ ]:
# formatted mode — easiest to read; shows operator tree with row estimates
card_txns.filter(col("is_fraud") == True) \
         .groupBy("category") \
         .agg(count("txn_id").alias("fraud_count")) \
         .explain(mode="formatted")

## Adaptive Query Execution (AQE)

AQE (enabled by default since Spark 3.2) re-optimises the query plan **at runtime** using statistics collected after each shuffle stage completes.

| Feature | What it does |
|---|---|
| **Coalescing shuffle partitions** | Merges small post-shuffle partitions to reduce task overhead |
| **Switching join strategies** | Converts sort-merge join → broadcast join when a table turns out small |
| **Skew join handling** | Splits oversized partitions to balance work across tasks |

In [ ]:
# Check AQE configuration
print("AQE enabled              :", spark.conf.get("spark.sql.adaptive.enabled"))
print("Coalesce partitions      :", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
print("Skew join handling       :", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
print("Shuffle partitions       :", spark.conf.get("spark.sql.shuffle.partitions"))

# With AQE, post-shuffle partition count can drop below shuffle.partitions
result = card_txns.groupBy("status").agg(count("txn_id").alias("n"))
result.show()
print(f"Post-AQE partitions: {result.rdd.getNumPartitions()}")

## Detecting and Fixing Skew

Data skew occurs when some partitions contain far more rows than others — one task takes much longer than the rest, stalling the whole stage. Common in banking data: a handful of high-volume customers dominate `customer_id` joins.

In [ ]:
from pyspark.sql.functions import spark_partition_id, count, col

# Measure partition skew after a groupBy
skew_check = (
    card_txns
    .groupBy("customer_id")
    .agg(count("txn_id").alias("txn_count"))
    .withColumn("pid", spark_partition_id())
    .groupBy("pid")
    .agg(
        count("customer_id").alias("keys_in_partition"),
        sum("txn_count").alias("total_rows"),
    )
    .orderBy(col("total_rows").desc())
)
skew_check.show()

# If skew is detected, use repartition by range to balance
balanced = (
    card_txns
    .groupBy("customer_id")
    .agg(count("txn_id").alias("txn_count"))
    .repartitionByRange(8, col("txn_count"))
)
print(f"Balanced partitions: {balanced.rdd.getNumPartitions()}")

In [ ]:
from pyspark.sql.functions import sum as _sum

# Also missing import fix above
from pyspark.sql.functions import sum as _sum

skew_check2 = (
    card_txns
    .withColumn("pid", spark_partition_id())
    .groupBy("pid")
    .agg(
        count("txn_id").alias("rows"),
    )
    .orderBy(col("rows").desc())
)
skew_check2.show()

## Tuning `spark.sql.shuffle.partitions`

The default of 200 shuffle partitions is sized for large clusters. On local mode or small datasets it creates hundreds of tiny tasks — set it to 2–4× your core count instead.

In [ ]:
# Compare execution with 200 vs 8 shuffle partitions on the same query
def run_agg(shuffle_parts):
    spark.conf.set("spark.sql.shuffle.partitions", str(shuffle_parts))
    result = card_txns.groupBy("customer_id").agg(count("txn_id").alias("n"))
    _ = result.count()  # trigger execution
    print(f"shuffle.partitions={shuffle_parts:>4}  →  output partitions: {result.rdd.getNumPartitions()}")

run_agg(200)
run_agg(8)

# Reset to sensible default for the rest of the session
spark.conf.set("spark.sql.shuffle.partitions", "8")

## Summary

- A partition is the unit of parallelism — one task per partition per core. Target 128–256 MB per partition.
- `repartition(n)` reshuffles evenly (full shuffle); `coalesce(n)` reduces without a shuffle (narrow).
- `repartition(n, col)` hash-partitions by column — all rows with the same key land together, eliminating a shuffle at join time.
- `partitionBy(col)` on write creates directory-per-value storage — downstream filters prune whole directories.
- Shuffles are triggered by `groupBy`, `join`, `distinct`, `orderBy`, `repartition`. Each shuffle writes to disk, transfers over the network, and deserialises on the other side.
- Look for `Exchange` nodes in `explain()` — each one is a shuffle stage boundary.
- Catalyst applies predicate pushdown (filter at read time) and column pruning (skip unread columns) automatically. Confirm via `explain(mode="extended")` and look at `PushedFilters` and `ReadSchema`.
- AQE dynamically coalesces small partitions, switches join strategies, and splits skewed partitions at runtime.
- Set `spark.sql.shuffle.partitions` to 2–4× core count for local workloads; the default of 200 is for large clusters.